# Telco Customer Churn Analysis
**DecodeLabs Data Analytics Assignment**  
**Author:** Matile Durin Mohwaduba  

---

## Business Problem

Customer churn — when a customer cancels their service — is one of the most costly problems a telecom company faces. Research consistently shows that acquiring a new customer costs **5 to 7 times more** than retaining an existing one.

This notebook analyses a real-world telecom dataset to answer the following business question:

> **Which customers are most likely to churn, and what factors are driving that behaviour?**

By understanding the patterns behind churn, the business can identify at-risk customers early and design targeted retention strategies — such as offering contract upgrades, personalised discounts, or proactive support outreach.

### Dataset Overview
The dataset is sourced from IBM's public Telco Customer Churn dataset. It contains **7,043 customer records** with 21 features covering:
- Customer demographics (gender, age category, dependents)
- Account information (tenure, contract type, billing method)
- Services subscribed (internet, phone, streaming, security)
- Financial data (monthly charges, total charges)
- Churn status (whether the customer left — Yes/No)

### Analytical Pipeline
This notebook follows a structured data analytics workflow:

| Step | Task |
|------|------|
| 1 | Data Collection |
| 2 | Data Inspection |
| 3 | Data Cleaning |
| 4 | Exploratory Data Analysis (EDA) |
| 5 | Predictive Modelling *(planned)* |

---

## Step 1: Data Collection

We begin by importing the required libraries and loading the dataset directly from IBM's public GitHub repository.

**Why load from a URL?**  
Loading from a stable public URL ensures reproducibility — anyone running this notebook will work with the exact same source data, regardless of their local environment. This is a best practice in collaborative and academic data work.

**Libraries used:**
- `pandas` — the core library for data manipulation and analysis in Python

In [ ]:
print("Hello World")

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print(df.head())

**Observation:** The dataset loaded successfully. We can already see a mix of categorical columns (Yes/No values) and numeric columns (tenure, MonthlyCharges, TotalCharges). Notice that `TotalCharges` appears to contain numeric values but will require closer inspection — this is flagged for the cleaning step.

---

## Step 2: Data Inspection

Before cleaning or analysing any dataset, we must understand its structure. This step answers four foundational questions:

1. How big is the dataset? (rows and columns)
2. What are the column names?
3. What data type is each column stored as?
4. Are there any missing values?

Skipping inspection is one of the most common beginner mistakes — it leads to cleaning decisions that don't match the actual data.

In [ ]:
print(df.shape)

**Observation:** The raw dataset contains **7,043 rows and 21 columns**. This is a medium-sized dataset — large enough to be statistically meaningful, small enough to work with efficiently in pandas.

In [ ]:
print(df.columns)

**Observation:** The 21 columns cover demographics, account details, subscribed services, charges, and the target variable `Churn`. No column names contain spaces or special characters that would cause issues in Python, though `customerID` will need to be excluded from any analysis as it is a unique identifier with no analytical value.

In [ ]:
print(df.dtypes)

**Observation — critical finding:** `TotalCharges` is stored as `object` (string), not as a numeric type. All other financial and numeric columns (`tenure`, `MonthlyCharges`) are correctly typed. This tells us that something inside the `TotalCharges` column is preventing pandas from reading it as a number. We will investigate and fix this in the cleaning step.

Additionally, `SeniorCitizen` is encoded as integers (0 and 1) rather than the Yes/No string format used by every other binary column. This inconsistency is noted for documentation purposes.

In [ ]:
print(df.isnull().sum())

**Observation:** The raw null count shows **zero nulls across all columns**. However, this does not mean the data is clean — as we already suspected from the dtype inspection, `TotalCharges` likely contains blank spaces that pandas is treating as valid string values rather than nulls. We will confirm and resolve this in the next step.

---

## Step 3: Data Cleaning

Data cleaning is the process of identifying and correcting errors, inconsistencies, and structural problems in the dataset before analysis begins. Analysing dirty data produces unreliable results — the classic principle of **"garbage in, garbage out"** applies directly here.

We identified two issues during inspection:
1. `TotalCharges` is stored as a string and contains blank spaces
2. Potential duplicate records

We address these systematically below.

### 3.1 Fix TotalCharges Data Type

**Root cause:** Customers with `tenure = 0` (brand new customers who have not yet received a bill) have a blank space `' '` in the `TotalCharges` column instead of a numeric value. When pandas loaded the CSV, it saw these blank spaces and defaulted the entire column to a string (object) type.

**Fix:** We use `pd.to_numeric()` with `errors='coerce'`, which converts the column to numeric and automatically turns any value that cannot be converted (the blank spaces) into `NaN`. This makes the missing values explicit and visible, allowing us to handle them deliberately in the next step.

In [ ]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'].replace(' ', pd.NA),
    errors='coerce'
)

### 3.2 Confirm and Handle Missing Values

Now that blank spaces have been converted to `NaN`, we re-run the null check to see the true missing value count.

In [ ]:
print(df.isnull().sum())

**Observation:** We now have **11 null values in `TotalCharges`** — these are the new customers with `tenure = 0` who have not yet been charged.

**Decision — drop rather than impute:** We drop these 11 rows for the following reasons:
- They represent only **0.16% of the dataset** — statistically negligible
- New customers with no billing history do not meaningfully contribute to churn analysis
- Their `TotalCharges` cannot be reliably inferred without additional business context

An alternative approach would be to impute `TotalCharges = MonthlyCharges` for these customers (since they've only been billed for one month), but given the negligible count, dropping is the more defensible choice here.

In [ ]:
df_clean = df.dropna()

print(df_clean.shape)

**Result:** After dropping the 11 null rows, we retain **7,032 rows** — 99.84% of the original dataset. No meaningful data has been lost.

### 3.3 Check for Duplicate Records

In [ ]:
duplicates = df_clean.duplicated().sum()

print("Duplicate Rows:", duplicates)

**Observation:** Zero duplicate rows detected. Each record represents a unique customer. No further action required.

### 3.4 Export Cleaned Dataset

We save the cleaned dataset to a CSV file for use in subsequent steps and future analyses. Setting `index=False` ensures the pandas row index is not written as an extra column in the file.

In [ ]:
df_clean.to_csv("Cleaned_Telco_Churn.csv", index=False)

print("Saved Successfully")

**Cleaning summary:**

| Issue | Finding | Action Taken |
|-------|---------|--------------|
| TotalCharges dtype | Stored as string due to blank spaces | Converted to float using `pd.to_numeric()` |
| Missing values | 11 nulls in TotalCharges (new customers) | Dropped — 0.16% of data |
| Duplicate records | 0 duplicates found | No action required |
| SeniorCitizen encoding | Integer (0/1) vs Yes/No strings | Documented — consistent within column |

The cleaned dataset is ready for analysis.

---

## Step 4: Exploratory Data Analysis (EDA)

Exploratory Data Analysis is the process of summarising and visualising the dataset to understand distributions, relationships, and patterns — before drawing any conclusions or building models.

EDA answers questions like:
- What does the data look like numerically?
- Is the target variable (Churn) balanced or skewed?
- Which features appear most strongly related to churn?

We begin with statistical summaries, then move to visualisations.

### 4.1 Statistical Summary

In [ ]:
df_clean.describe()

**Observations from the statistical summary:**

- **`tenure`** ranges from 1 to 72 months (mean ≈ 32 months). The wide spread suggests a mix of very new and very long-term customers — tenure is likely a strong predictor of churn.
- **`MonthlyCharges`** ranges from \$18.25 to \$118.75 (mean ≈ \$64.80). The spread indicates significant variation in plan pricing across the customer base.
- **`TotalCharges`** ranges from \$18.80 to \$8,684.80 — this wide range is expected since TotalCharges is a product of tenure and MonthlyCharges. Long-tenured, high-paying customers will have very high totals.
- **`SeniorCitizen`** has a mean of 0.16, meaning approximately **16% of customers are senior citizens** — a minority segment worth examining separately.

### 4.2 Deeper Inspection of Key Numeric Columns

In [ ]:
print(df_clean['MonthlyCharges'].describe())

In [ ]:
print(df_clean['tenure'].describe())

**Observation:** The median tenure is 29 months while the mean is 32 months — these are close, suggesting no extreme skew in tenure distribution. For MonthlyCharges, the median (\$70.35) is slightly above the mean (\$64.80), which may indicate a slight left skew — more customers on higher-priced plans than the average suggests.

### 4.3 Churn Distribution

Before exploring relationships between variables, we must understand the balance of our target variable. If churn is extremely rare or extremely common, that affects how we interpret patterns and would affect any future modelling approach.

In [ ]:
print(df_clean['Churn'].value_counts())

**Observation:** The dataset shows a **73% / 27% split** — 5,163 customers did not churn, 1,869 did. This is a moderately imbalanced dataset. The imbalance is not severe enough to invalidate analysis, but it is important to note for future modelling: a naive classifier that predicts "No churn" for every customer would achieve 73% accuracy while being completely useless. Any predictive model built on this data should use appropriate metrics (precision, recall, F1-score, ROC-AUC) rather than raw accuracy.

### 4.4 Visualisation: Churn Distribution

We visualise the churn split to make the imbalance immediately readable.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='Churn', data=df_clean)

plt.title("Customer Churn Distribution")
plt.show()

**Observation:** The bar chart confirms the 73/27 churn split visually. The majority of customers are retained, but the churned segment (1,869 customers) is large enough to analyse meaningfully.

### 4.5 Visualisation: Contract Type vs Churn

Contract type is a strong candidate for a churn driver — customers on flexible month-to-month plans have far less friction to leave than those locked into annual or two-year contracts. We examine this relationship directly.

In [ ]:
sns.countplot(
    x='Contract',
    hue='Churn',
    data=df_clean
)

plt.title("Contract Type vs Churn")
plt.show()

**Observation — key business insight:** Month-to-month customers churn at a dramatically higher rate than customers on one or two-year contracts. Customers on two-year contracts show almost negligible churn by comparison.

**Business implication:** Contract type is likely one of the strongest predictors of churn in this dataset. A targeted retention strategy might focus on converting month-to-month customers to longer-term contracts through incentives — for example, offering a discounted annual plan to customers who have been on month-to-month billing for more than 6 months without churning.

---

## Step 5: Predictive Modelling

> ⚠️ **Work in Progress** — This section is planned but not yet implemented.

### Planned Approach

Based on the EDA findings, the following modelling pipeline is intended:

**Feature Engineering**
- Encode categorical variables (Label Encoding or One-Hot Encoding)
- Standardise numeric features (tenure, MonthlyCharges, TotalCharges) using `StandardScaler`
- Address class imbalance using SMOTE or class weighting

**Model Selection**
- Baseline: Logistic Regression (interpretable, good for binary classification)
- Comparison: Random Forest Classifier (handles non-linear relationships)

**Evaluation Metrics**
- Accuracy, Precision, Recall, F1-Score, ROC-AUC
- Confusion Matrix

**Expected Output**
- A trained model that predicts churn probability for individual customers
- Feature importance ranking to identify the top drivers of churn

---

*This section will be completed in the next iteration of this notebook.*